In [9]:
import pandas as pd
import rdata
import numpy as np
import tensorflow as tf

from pathlib import Path
from urllib.request import urlretrieve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from tensorflow import keras
from tensorflow.keras import layers, regularizers


In [12]:
import pandas as pd
import rdata
import numpy as np
import tensorflow as tf

from pathlib import Path
from urllib.request import urlretrieve
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from tensorflow import keras
from tensorflow.keras import layers, regularizers

# ============================================================
# 1. CARGA DE TODOS LOS DATOS (BARCELONA, MADRID, VALENCIA)
# ============================================================
DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
BASE_URL = "https://raw.githubusercontent.com/emartincar21/UT05_Proyecto_Final/main/data"

# Restauramos el diccionario original para que descargue todo sin errores
SALE_FILES = {
    "Madrid": "Madrid_Sale.rda",
    "Barcelona": "Barcelona_Sale.rda",
    "Valencia": "Valencia_Sale.rda"
}

def download_file(filename: str) -> Path:
    local_path = DATA_DIR / filename
    if not local_path.exists():
        url = f"{BASE_URL}/{filename}"
        print(f"Descargando {filename}...")
        urlretrieve(url, local_path)
    return local_path

def load_rda(filename: str) -> dict:
    local_path = download_file(filename)
    return rdata.read_rda(str(local_path), default_encoding="utf8")

def load_city_sales(city_name: str, filename: str) -> pd.DataFrame:
    data = load_rda(filename)
    df_name = list(data.keys())[0]
    df = data[df_name]
    df["CITY"] = city_name
    return df

def load_all_sales() -> pd.DataFrame:
    city_dfs = []
    for city_name, filename in SALE_FILES.items():
        print(f"Procesando datos de {city_name}...")
        df_city = load_city_sales(city_name, filename)
        city_dfs.append(df_city)
    return pd.concat(city_dfs, ignore_index=True)

# Descargamos y unimos todo en memoria
df_completo = load_all_sales()

# ============================================================
# 2. FILTRADO: SOLO MADRID Y VARIABLES SELECCIONADAS
# ============================================================
print("\nFiltrando el dataset: Aislando únicamente Madrid...")
df_madrid = df_completo[df_completo["CITY"] == "Madrid"].copy()

target = "PRICE"
# Variables seleccionadas (Sin CITY ni Distancias)
features = [
    "CONSTRUCTEDAREA", "ROOMNUMBER", "BATHNUMBER", "HASTERRACE", "HASLIFT",
    "HASAIRCONDITIONING", "HASPARKINGSPACE", "HASBOXROOM", "HASWARDROBE",
    "HASSWIMMINGPOOL", "HASGARDEN", "ISDUPLEX", "ISSTUDIO", "ISINTOPFLOOR",
    "LATITUDE", "LONGITUDE"
]

# Nos quedamos solo con las columnas deseadas y eliminamos nulos
df_model = df_madrid[features + [target]].copy().dropna()
X = df_model[features]

# Aplicamos escala logarítmica al target (Precio) para mejorar el aprendizaje
y = np.log1p(df_model[target]) 

print(f"Datos listos. Filas finales para entrenar: {len(df_model)}")

# ============================================================
# 3. SPLIT Y ESCALADO
# ============================================================
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.1765, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# ============================================================
# 4. FUNCIÓN PARA CREAR Y ENTRENAR MODELOS
# ============================================================
def entrenar_experimento(nombre_experimento, dropout_rate, es_pro=False):
    print(f"\n{'='*50}")
    print(f"🚀 INICIANDO EXPERIMENTO: {nombre_experimento}")
    print(f"{'='*50}")
    
    input_dim = X_train_scaled.shape[1]
    
    if es_pro:
        # ARQUITECTURA PRO
        model = keras.Sequential([
            layers.Input(shape=(input_dim,)),
            layers.Dense(128, kernel_regularizer=regularizers.l2(0.001)),
            layers.BatchNormalization(),
            layers.Activation("relu"),
            layers.Dropout(dropout_rate),
            
            layers.Dense(64, kernel_regularizer=regularizers.l2(0.001)),
            layers.BatchNormalization(),
            layers.Activation("relu"),
            layers.Dropout(dropout_rate),
            
            layers.Dense(32, kernel_regularizer=regularizers.l2(0.001)),
            layers.BatchNormalization(),
            layers.Activation("relu"),
            
            layers.Dense(1) 
        ])
    else:
        # ARQUITECTURA BÁSICA (PDF)
        model = keras.Sequential([
            layers.Input(shape=(input_dim,)),
            layers.Dense(128, activation="relu"),
            layers.Dropout(dropout_rate),
            layers.Dense(64, activation="relu"),
            layers.Dropout(dropout_rate),
            layers.Dense(32, activation="relu"),
            layers.Dense(1)
        ])

    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    
    tensorboard_cb = keras.callbacks.TensorBoard(log_dir=f"./logs_madrid/{nombre_experimento}")
    checkpoint_cb = keras.callbacks.ModelCheckpoint(f"mejor_modelo_{nombre_experimento}.keras", save_best_only=True, monitor="val_loss")
    
    callbacks_list = [tensorboard_cb, checkpoint_cb]
    
    epocas = 50
    if es_pro:
        epocas = 150 
        reduce_lr_cb = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
        early_stop_cb = keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1)
        callbacks_list.extend([reduce_lr_cb, early_stop_cb])

    model.fit(
        X_train_scaled, y_train,
        validation_data=(X_val_scaled, y_val),
        epochs=epocas,
        batch_size=64,
        callbacks=callbacks_list,
        verbose=1 
    )
    
    return model

# ============================================================
# 5. EJECUCIÓN DE LA BATERÍA DE EXPERIMENTOS
# ============================================================
# Los 3 obligatorios para la memoria
dropouts_basicos = [0.2, 0.5, 0.8]

for d_rate in dropouts_basicos:
    entrenar_experimento(nombre_experimento=f"basico_dropout_{d_rate}", dropout_rate=d_rate, es_pro=False)

# El 4º Experimento PRO
modelo_pro = entrenar_experimento(nombre_experimento="PRO_version", dropout_rate=0.2, es_pro=True)

# ============================================================
# 6. EVALUACIÓN FINAL DEL MODELO PRO CONTRA EL TEST CIEGO
# ============================================================
print("\n" + "="*50)
print("🏆 EVALUACIÓN DEL MEJOR MODELO (VERSIÓN PRO) 🏆")
print("="*50)

mejor_pro = keras.models.load_model("mejor_modelo_PRO_version.keras")
y_pred_log = mejor_pro.predict(X_test_scaled)

# Revertimos el logaritmo
y_pred_real = np.expm1(y_pred_log).flatten()
y_test_real = np.expm1(y_test).to_numpy()

mae_euros_real = mean_absolute_error(y_test_real, y_pred_real)
print(f"\nTEST MAE (Mundo Real): El modelo PRO se equivoca de media por {mae_euros_real:,.2f} €")


Procesando datos de Madrid...
Descargando Madrid_Sale.rda...


HTTPError: HTTP Error 404: Not Found

In [ ]:
# ============================================================
# 3. FUNCIÓN PARA CREAR Y ENTRENAR MODELOS
# ============================================================
def entrenar_experimento(nombre_experimento, dropout_rate, es_pro=False):
    print(f"\n{'='*50}")
    print(f"🚀 INICIANDO EXPERIMENTO: {nombre_experimento}")
    print(f"{'='*50}")
    
    input_dim = X_train_scaled.shape[1]
    
    if es_pro:
        # ARQUITECTURA PRO: Regularización L2, Batch Normalization y Dropout moderado
        model = keras.Sequential([
            layers.Input(shape=(input_dim,)),
            layers.Dense(128, kernel_regularizer=regularizers.l2(0.001)),
            layers.BatchNormalization(),
            layers.Activation("relu"),
            layers.Dropout(dropout_rate),
            
            layers.Dense(64, kernel_regularizer=regularizers.l2(0.001)),
            layers.BatchNormalization(),
            layers.Activation("relu"),
            layers.Dropout(dropout_rate),
            
            layers.Dense(32, kernel_regularizer=regularizers.l2(0.001)),
            layers.BatchNormalization(),
            layers.Activation("relu"),
            
            layers.Dense(1) 
        ])
    else:
        # ARQUITECTURA BÁSICA (La que exige el PDF para comparar Dropout)
        model = keras.Sequential([
            layers.Input(shape=(input_dim,)),
            layers.Dense(128, activation="relu"),
            layers.Dropout(dropout_rate),
            layers.Dense(64, activation="relu"),
            layers.Dropout(dropout_rate),
            layers.Dense(32, activation="relu"),
            layers.Dense(1)
        ])

    model.compile(optimizer="adam", loss="mse", metrics=["mae"])
    
    # Callbacks base
    tensorboard_cb = keras.callbacks.TensorBoard(log_dir=f"./logs_madrid/{nombre_experimento}")
    checkpoint_cb = keras.callbacks.ModelCheckpoint(f"mejor_modelo_{nombre_experimento}.keras", save_best_only=True, monitor="val_loss")
    
    callbacks_list = [tensorboard_cb, checkpoint_cb]
    
    # Si es el modelo PRO, le damos superpoderes (Early Stopping y Reduce LR)
    epocas = 50
    if es_pro:
        epocas = 150 # Le damos más épocas porque Early Stopping lo detendrá si deja de mejorar
        reduce_lr_cb = keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=5, min_lr=1e-6, verbose=1)
        early_stop_cb = keras.callbacks.EarlyStopping(monitor='val_loss', patience=15, restore_best_weights=True, verbose=1)
        callbacks_list.extend([reduce_lr_cb, early_stop_cb])
    # Entrenamiento
    model.fit(
        X_train_scaled, y_train,
        validation_data=(X_val_scaled, y_val),
        epochs=epocas,
        batch_size=64,
        callbacks=callbacks_list,
        verbose=1 # Cambia a 0 si no quieres que te llene la pantalla de texto
    )
    
    return model

In [ ]:

# ============================================================
# 4. EJECUCIÓN DE LA BATERÍA DE EXPERIMENTOS
# ============================================================
# Experimentos obligatorios para la memoria (Regla 2 del PDF)
dropouts_basicos = [0.2, 0.5, 0.8]

for d_rate in dropouts_basicos:
    entrenar_experimento(nombre_experimento=f"basico_dropout_{d_rate}", dropout_rate=d_rate, es_pro=False)

# El 4º Experimento: La joya de la corona
modelo_pro = entrenar_experimento(nombre_experimento="PRO_version", dropout_rate=0.2, es_pro=True)

# ============================================================
# 5. EVALUACIÓN FINAL DEL MODELO PRO CONTRA EL TEST CIEGO
# ============================================================
print("\n" + "="*50)
print("🏆 EVALUACIÓN DEL MEJOR MODELO (VERSIÓN PRO) 🏆")
print("="*50)

# Cargamos el mejor peso guardado del modelo PRO
mejor_pro = keras.models.load_model("mejor_modelo_PRO_version.keras")

y_pred_log = mejor_pro.predict(X_test_scaled)

# Revertimos el logaritmo para hablar en Euros
y_pred_real = np.expm1(y_pred_log).flatten()
y_test_real = np.expm1(y_test).to_numpy()

mae_euros_real = mean_absolute_error(y_test_real, y_pred_real)
print(f"\nTEST MAE (Mundo Real): El modelo PRO se equivoca de media por {mae_euros_real:,.2f} €")
